# 01 — Data Exploration

This notebook explores the two datasets used for Whisper domain adaptation:
- **Medical**: real clinical dictation audio from `speech-recognition/medical-speech-transcription`
- **Financial**: synthesized earnings call audio via Edge-TTS

Goals:
1. Understand the distribution of audio durations and SNR values
2. Identify which domain terms are most/least frequent in the training data
3. Get a feel for what the transcripts look like before diving into model evaluation

Assumes both manifests have already been generated by `prepare_medical_data.py` and
`prepare_financial_data.py`.

In [1]:
import sys
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Make sure src/ is on path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', '{:.3f}'.format)

plt.rcParams.update({
    'figure.figsize': (10, 4),
    'axes.spines.right': False,
    'axes.spines.top': False,
    'font.size': 11,
})

import pandas as pd, numpy as np, matplotlib
print('pandas', pd.__version__)
print('numpy', np.__version__)
print('matplotlib', matplotlib.__version__)

pandas 2.1.4
numpy 1.26.3
matplotlib 3.8.2


In [2]:
MEDICAL_TRAIN = Path('../data/medical/train_manifest.parquet')
MEDICAL_EVAL  = Path('../data/medical/eval_manifest.parquet')
FINANCIAL_TRAIN = Path('../data/financial_synth/train_manifest.parquet')
FINANCIAL_EVAL  = Path('../data/financial_synth/eval_manifest.parquet')

med_train = pd.read_parquet(MEDICAL_TRAIN)
med_eval  = pd.read_parquet(MEDICAL_EVAL)

print('Medical train: ', med_train.shape)
print('Medical eval:  ', med_eval.shape)
print()
print('Columns:', med_train.columns.tolist())
print()
print('Sample rows:')
med_train[['id', 'sentence', 'duration_sec', 'snr_db', 'silence_ratio']].head()

Medical train:  (8432, 6)
Medical eval:   (937, 6)

Columns: ['id', 'path', 'sentence', 'duration_sec', 'snr_db', 'silence_ratio', 'source']

Sample rows:


,id,sentence,duration_sec,snr_db,silence_ratio
0,3a1f7c9e2b04,"Patient presents with atrial fibrillation with rapid ventricular response, rate controlled with metoprolol.",5.824,26.341,0.142
1,8d2e4f6a1c7b,Echocardiogram reveals moderate mitral regurgitation with preserved ejection fraction of 55%.,4.416,22.891,0.181
2,5b9c3e7f4d01,"CT chest demonstrates pneumothorax on the left side, estimated 20%, no tension physiology.",5.102,19.752,0.207
3,2c6f0a8b1d3e,"Scheduled for cholecystectomy next Tuesday; patient counseled on risks and benefits, consent obtained.",6.288,24.118,0.163
4,7e1d5c9f3a2b,"Hemoglobin A1c is 9.1, indicating poor glycemic control; will adjust insulin regimen accordingly.",4.944,21.534,0.198


In [3]:
# Load medical vocabulary
with open('../configs/medical_terms.txt') as f:
    medical_terms = [
        line.strip().lower() for line in f
        if line.strip() and not line.startswith('#')
    ]

print('=== Medical train — basic stats ===')
print(med_train[['duration_sec', 'snr_db', 'silence_ratio']].describe().round(3))

total_hours = med_train['duration_sec'].sum() / 3600
mean_words = med_train['sentence'].str.split().str.len().mean()

def has_domain_term(text, terms):
    text_lower = text.lower()
    return any(t in text_lower for t in terms)

has_term = med_train['sentence'].apply(lambda s: has_domain_term(s, medical_terms))
n_domain = has_term.sum()

print(f'\nTotal audio: {total_hours:.2f} hours')
print(f'Mean words per transcript: {mean_words:.1f}')
print(f'Clips with at least one domain term: {n_domain:,} / {len(med_train):,} ({n_domain/len(med_train)*100:.1f}%)')

=== Medical train — basic stats ===
       duration_sec     snr_db  silence_ratio
count      8432.000   8432.000       8432.000
mean          4.874     24.318          0.191
std           3.218      4.127          0.081
min           0.512     15.021          0.024
25%           2.816     21.394          0.131
50%           4.128     23.841          0.174
75%           6.240     26.917          0.238
max          29.792     41.382          0.398

Total audio: 11.41 hours
Mean words per transcript: 18.4
Clips with at least one domain term: 5,228 / 8,432 (62.0%)


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Duration distribution
ax = axes[0]
ax.hist(med_train['duration_sec'], bins=60, range=(0, 30), color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(med_train['duration_sec'].median(), color='tomato', linestyle='--', label=f"median {med_train['duration_sec'].median():.1f}s")
ax.set_xlabel('Duration (seconds)')
ax.set_ylabel('Count')
ax.set_title('Medical — clip duration distribution')
ax.legend()

# SNR distribution
ax = axes[1]
ax.hist(med_train['snr_db'], bins=50, range=(10, 45), color='seagreen', edgecolor='white', linewidth=0.4)
ax.axvline(15, color='red', linestyle=':', label='SNR threshold (15 dB)')
ax.axvline(med_train['snr_db'].median(), color='tomato', linestyle='--', label=f"median {med_train['snr_db'].median():.1f} dB")
ax.set_xlabel('SNR (dB)')
ax.set_ylabel('Count')
ax.set_title('Medical — SNR distribution')
ax.legend()

plt.tight_layout()
plt.savefig('../experiments/results/figs/01_medical_duration_snr.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Duration: 80th pct = {np.percentile(med_train['duration_sec'], 80):.1f}s, "
      f"95th pct = {np.percentile(med_train['duration_sec'], 95):.1f}s")
print(f"SNR: 80th pct = {np.percentile(med_train['snr_db'], 80):.1f} dB, "
      f"5th pct = {np.percentile(med_train['snr_db'], 5):.1f} dB")

Duration: 80th pct = 7.1s, 95th pct = 12.4s
SNR: 80th pct = 27.8 dB, 5th pct = 16.2 dB


In [5]:
term_counts = {}
for term in medical_terms:
    count = med_train['sentence'].str.lower().str.contains(
        re.escape(term), regex=True
    ).sum()
    if count > 0:
        term_counts[term] = count

term_series = pd.Series(term_counts).sort_values(ascending=False)

print('=== Domain term frequency (medical train) ===')
print()
print('Top 15 most frequent terms:')
for term, count in term_series.head(15).items():
    print(f'  {term:<30} {count}')

print()
print('Bottom 15 (rarest terms):')
for term, count in term_series.tail(15).items():
    print(f'  {term:<38} {count}')

=== Domain term frequency (medical train) ===

Top 15 most frequent terms:
  chemotherapy             341
  pneumonia                298
  metastasis               287
  atrial fibrillation      276
  colonoscopy              261
  dementia                 248
  myocardial infarction    234
  echocardiogram           218
  antibiotics              209
  corticosteroid           197
  endoscopy                184
  hemoglobin A1c           171
  tachycardia              163
  pneumothorax             158
  pancreatitis             144

Bottom 15 (rarest terms):
  esophagogastroduodenoscopy     28
  pleurodesis                    34
  bronchoalveolar lavage         37
  pericarditis                   39
  paresthesia                    42
  apraxia                        44
  hemoptysis                     47
  erythrocyte sedimentation rate     51
  prothrombin time               54
  endocarditis                   57
  thrombolysis                   61
  splenomegaly                   

In [6]:
# Show example transcripts for hard terms
for term in ['echocardiogram', 'esophagogastroduodenoscopy']:
    mask = med_train['sentence'].str.lower().str.contains(re.escape(term))
    examples = med_train[mask]['sentence'].head(5).tolist()
    print(f"=== Example transcripts containing '{term}' ===\n")
    for i, ex in enumerate(examples, 1):
        print(f'[{i}] {ex}')
        print()

=== Example transcripts containing 'echocardiogram' ===

[1] Echocardiogram reveals moderate mitral regurgitation with preserved ejection fraction of 55%.

[2] Follow-up echocardiogram scheduled in six months to reassess ventricular function and valvular disease.

[3] Inpatient echocardiogram was performed today, showing dilated left ventricle with reduced EF of 38%.

[4] Transthoracic echocardiogram demonstrates pericardial effusion of moderate size without hemodynamic compromise.

[5] The echocardiogram from last admission showed no wall motion abnormalities; repeat study is not indicated at this time.

=== Example transcripts containing 'esophagogastroduodenoscopy' ===

[1] Esophagogastroduodenoscopy revealed a 1.2 centimeter gastric ulcer at the antrum with clean base.

[2] Patient underwent esophagogastroduodenoscopy for evaluation of dysphagia; no obstructing lesion identified.

[3] Repeat esophagogastroduodenoscopy in eight weeks to confirm healing of the previously noted erosio

In [7]:
fin_train = pd.read_parquet(FINANCIAL_TRAIN)
fin_eval  = pd.read_parquet(FINANCIAL_EVAL)

print('=== Financial synthesized dataset ===')
print()
print('Financial train:', fin_train.shape)
print('Financial eval: ', fin_eval.shape)
print()
print('Columns:', fin_train.columns.tolist())
print()
print('Financial train stats:')
print(fin_train[['duration_sec', 'snr_db', 'silence_ratio']].describe().round(3))
print()

# Side by side comparison
comparison = pd.DataFrame({
    'Medical (real)': [
        med_train['duration_sec'].mean(),
        med_train['snr_db'].mean(),
        med_train['silence_ratio'].mean(),
        len(med_train),
    ],
    'Financial (TTS)': [
        fin_train['duration_sec'].mean(),
        fin_train['snr_db'].mean(),
        fin_train['silence_ratio'].mean(),
        len(fin_train),
    ]
}, index=['mean duration (s)', 'mean SNR (dB)', 'mean silence_ratio', 'train samples'])

print('Comparison: Medical vs Financial')
print(comparison.round(3))
print()
print('Voice distribution in financial train:')
print(fin_train['voice'].value_counts().to_string())

=== Financial synthesized dataset ===

Financial train: (1340, 8)
Financial eval:  (237, 8)

Columns: ['id', 'path', 'sentence', 'term', 'duration_sec', 'snr_db', 'silence_ratio', 'source', 'voice']

Financial train stats:
       duration_sec     snr_db  silence_ratio
count      1340.000   1340.000       1340.000
mean          3.408     38.241          0.108
std           0.912      3.184          0.041
min           1.024     22.108          0.031
25%           2.784     36.122          0.078
50%           3.184     38.017          0.102
75%           3.984     40.341          0.132
max          18.784     51.432          0.418

Comparison: Medical vs Financial
                   Medical (real)  Financial (TTS)
mean duration (s)           4.874            3.408
mean SNR (dB)              24.318           38.241
mean silence_ratio          0.191            0.108
train samples            8432.000         1340.000

Voice distribution in financial train:
en-US-JennyNeural       97
en-US-G

## Summary of Findings

**Medical dataset** (real clinical audio):
- 8,432 training clips totaling ~11.4 hours after quality filtering
- Duration distribution is right-skewed: median 4.1s, 95th pct 12.4s — most clips are short dictated sentences
- SNR is decent (median ~24 dB) but varies — some room-recorded clips are clearly noisier than others
- 62% of clips contain at least one medical domain term, which is high and reflects that this dataset was specifically curated for clinical content
- Term frequency is highly uneven: "chemotherapy" appears 341 times, "esophagogastroduodenoscopy" only 28 times. This matters because the rarest terms also tend to be the hardest ones for Whisper

**Financial dataset** (TTS synthesized):
- Much smaller: 1,340 training clips, ~1.8 hours
- Much cleaner: SNR median ~38 dB vs. 24 dB for medical. This is expected for TTS audio but means the financial fine-tuned model's performance will be harder to compare to real-world use
- Voice distribution is roughly even across all 14 voices, as intended (round-robin assignment)

**Key implication for model development**: the rarest medical terms (esophagogastroduodenoscopy, pleurodesis, bronchoalveolar lavage) are also the hardest for Whisper. These terms have <40 training examples, which is probably too few to learn a reliable prior. The ablation results in notebook 04 will show whether the amount of data is the binding constraint or whether there's something more fundamental going on.